# Auliya G1/G2 diagnosis — E1 and optional E2

Attach the extracted auliya-robusta-sni dataset AND the previous full Controlled Generalization notebook output (both RGB last.pt checkpoints and reports). Default E1 only: frozen localization/class ranking, no training. Optional independent HAAR training: RUN_E1=False, RUN_HAAR=True. Group A remains locked. GPU and internet enabled.


In [ ]:
# Settings
SCIENTIFIC_CODE_COMMIT = "c4cd7a66dfd087325a9d32b193b07f199bfbbca4"
RUN_E1 = True
RUN_HAAR = False
AUTHORIZE_TRAINING = RUN_HAAR
GROUPS = ["B", "C"]  # or just ["B"] / ["C"] in separate sessions
DEVICE = "0"
DEVELOPMENT_OVERRIDE = ""  # optional exact mounted development directory
METADATA_OVERRIDE = ""     # optional mounted official metadata ZIP
import hashlib, json, os, shutil, subprocess, sys
from pathlib import Path
from urllib.request import urlopen

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
PROJECT = WORK / "auliya-gc-diagnosis-v1"
REPO = WORK / "coffee-bean-detection-gc-diagnosis"
DATA = WORK / "auliya-gc-diagnosis-data"
DOWNLOAD = WORK / "auliya-gc-diagnosis-assets"
assert INPUT.is_dir() and WORK.is_dir(), "Run on Kaggle"
assert GROUPS and len(set(GROUPS))==len(GROUPS) and set(GROUPS)<={"B","C"}
DOWNLOAD.mkdir(exist_ok=True)
DEV_SHA = "23d653ca2e98c185a76bb7b2095d6afc6401876cbe48fd219cb42bdc7ecbd360"
META_SHA = "798bb53b0cfad35dfcfbcc463fbcb79ff4af61f9807803d6bcc65d37f7c8feaa"
MODEL_SHA = "3da1d83e31caec96f9300eb4064f4f62882c133c7c264d63dfe61a7c197837a4"

def digest(p):
    h = hashlib.sha256()
    with Path(p).open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

if DEVELOPMENT_OVERRIDE:
    devs = [Path(DEVELOPMENT_OVERRIDE).resolve()]
else:
    devs = sorted(p.parent for p in INPUT.rglob("development_contract.json")
                  if p.parent.name == "development" and digest(p) == DEV_SHA)
if len(devs) != 1:
    raise RuntimeError(f"Expected exactly one verified development folder; found {devs}")
DEVELOPMENT = devs[0]
if not DEVELOPMENT.resolve().is_relative_to(INPUT.resolve()) or digest(DEVELOPMENT/"development_contract.json") != DEV_SHA:
    raise RuntimeError("Invalid mounted development contract")
print("Development:", DEVELOPMENT, flush=True)

prior = sorted(p for p in INPUT.rglob(PROJECT.name) if p.is_dir() and (p/"logs").is_dir())
if len(prior)>1:
    raise RuntimeError("Attach at most one previous Notebook Output for resume")
if prior and not PROJECT.exists():
    shutil.copytree(prior[0],PROJECT)
elif prior:
    print("Using current working output; not overwriting with prior")
PROJECT.mkdir(exist_ok=True)
(PROJECT/"logs").mkdir(exist_ok=True)

def fetch_verified(url, destination, expected):
    if destination.is_file() and digest(destination)==expected:
        return destination
    tmp = destination.with_suffix(".part")
    try:
        with urlopen(url,timeout=120) as source, tmp.open("wb") as sink:
            shutil.copyfileobj(source,sink)
        if digest(tmp)!=expected:
            raise RuntimeError(f"SHA mismatch: {destination}")
        tmp.replace(destination)
    finally:
        tmp.unlink(missing_ok=True)
    return destination

def run(cmd,log=None,cwd=None):
    cmd=list(map(str,cmd))
    print("$ "+" ".join(cmd),flush=True)
    if log is None:
        subprocess.run(cmd,check=True,cwd=cwd)
        return
    with Path(log).open("a",encoding="utf-8") as stream:
        with subprocess.Popen(cmd,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as p:
            for line in p.stdout:
                print(line,end="",flush=True)
                stream.write(line)
                stream.flush()
            if p.wait()!=0:
                raise RuntimeError(f"Command failed; see {log}")

if METADATA_OVERRIDE:
    METADATA=Path(METADATA_OVERRIDE).resolve()
    if not METADATA.is_relative_to(INPUT.resolve()) or digest(METADATA)!=META_SHA:
        raise RuntimeError("Official metadata override invalid")
else:
    METADATA=fetch_verified("https://zenodo.org/api/records/24346584/files/splits_metadata_results_scripts.zip/content",DOWNLOAD/"metadata.zip",META_SHA)
WEIGHTS=fetch_verified("https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo26s-seg.pt",DOWNLOAD/"yolo26s-seg.pt",MODEL_SHA)
print("Verified metadata and pretrained weights.")

# Exact historical RGB result/checkpoint pairs, not the report-only ZIP.
EXPECTED = {"B":"38385effe9335bf5ad23fee9913f19f5373494da1c20bdb578e1138ae52719fc",
            "C":"41ffe7f52503d64ca4a008dd477e0575aa52604c8274f5c7005f1fb0a0723642"}
candidates = []
for p in INPUT.rglob("validation_result.json"):
    if p.parent.name!="B": continue
    root=p.parent.parent
    try:
        reports={g:json.loads((root/g/"validation_result.json").read_text()) for g in "BC"}
        if all(reports[g]["checkpoint_sha256"]==EXPECTED[g] for g in "BC"):
            if not RUN_E1 or all((root/g/"weights/last.pt").is_file() and digest(root/g/"weights/last.pt")==EXPECTED[g] for g in "BC"):
                candidates.append(root)
    except (KeyError,FileNotFoundError,ValueError): pass
if len(candidates)!=1:
    raise RuntimeError(f"Attach exactly one original G1/G2 full notebook output; found {candidates}")
RGB_RUNS=candidates[0]
assert RUN_E1 or RUN_HAAR
print("Historical RGB:",RGB_RUNS)


In [ ]:
# Pinned scientific code, no package import from bundled Kaggle data.
if not REPO.exists():
    run(["git","clone","--quiet","--no-checkout","https://github.com/ediprin/coffee-bean-detection.git",REPO])
if subprocess.run(["git","-C",str(REPO),"cat-file","-e",SCIENTIFIC_CODE_COMMIT+"^{commit}"],capture_output=True).returncode!=0:
    run(["git","-C",REPO,"fetch","--quiet","origin","codex/auliya-controlled-generalization"])
run(["git","-C",REPO,"checkout","--quiet","--detach",SCIENTIFIC_CODE_COMMIT])
assert subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip()==SCIENTIFIC_CODE_COMMIT
os.environ["PYTHONPATH"]=str(REPO/"src")+os.pathsep+os.environ.get("PYTHONPATH","")
os.environ["PYTHONUNBUFFERED"]="1"
os.environ["OMP_NUM_THREADS"]="2"
# Official fold intentionally uses newer Ultralytics than the pilot.
run([sys.executable,"-m","pip","install","-q","ultralytics==8.4.210","scikit-learn","pytest"])
# Use subprocess to avoid stale imports from earlier Kaggle cells.
runtime=json.loads(subprocess.check_output([sys.executable,"-c",
    "import json,torch,ultralytics,numpy,cv2,sklearn,platform; print(json.dumps(dict(torch=str(torch.__version__),ultralytics=ultralytics.__version__,numpy=numpy.__version__,opencv=cv2.__version__,sklearn=sklearn.__version__,python=platform.python_version(),gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)))"],text=True).strip().splitlines()[-1])
assert runtime["ultralytics"]=="8.4.210",runtime
if (RUN_E1 or RUN_HAAR) and runtime["gpu"] is None:
    raise RuntimeError("Enable Kaggle GPU")
runtime["code_commit"]=SCIENTIFIC_CODE_COMMIT
marker=PROJECT/"environment.json"
if marker.exists() and json.loads(marker.read_text())!=runtime:
    raise RuntimeError("Runtime changed; do not resume in a different environment")
marker.write_text(json.dumps(runtime,indent=2))
print(runtime)
run([sys.executable,"-m","pip","freeze"],log=PROJECT/"logs/pip-freeze.txt")
run([sys.executable,"-m","pytest","-q","tests/test_auliya_official_fold.py","tests/test_auliya_representation_audit.py","tests/test_auliya_controlled_generalization.py","tests/test_auliya_gc_error_audit.py"],cwd=REPO,log=PROJECT/"logs/tests.log")


In [ ]:
# E1: no training; both exact historical RGB models on both targets.
if RUN_E1:
    run([sys.executable,"-u","-m","coffee_detector.analysis.auliya_gc_error_audit",
         "--development",DEVELOPMENT,"--metadata",METADATA,"--data",DATA,
         "--checkpoint-b",RGB_RUNS/"B/weights/last.pt","--checkpoint-c",RGB_RUNS/"C/weights/last.pt",
         "--output",PROJECT/"E1","--device","cuda:0"],cwd=REPO,log=PROJECT/"logs/E1.log")
    e1=json.loads((PROJECT/"E1/report.json").read_text())
    assert e1["test_A_images_accessed"] is False and e1["training_executed"] is False
    for case,rows in e1["results"].items():
        print(case,rows["iou0.5_conf0.25_clean18"],rows["iou0.5_conf0.25_hard5"])


In [ ]:
# E2 optional: same Haar as official fold, no new operator.
if RUN_HAAR:
    for group in GROUPS:
        run([sys.executable,"-u","-m","coffee_detector.experiments.run_auliya_gc_haar",
             "--development",DEVELOPMENT,"--metadata",METADATA,"--data",DATA,
             "--weights",WEIGHTS,"--output",PROJECT/"HAAR","--rgb-results",RGB_RUNS,
             "--device",DEVICE,"--group",group,"--authorize-training"],
            cwd=REPO,log=PROJECT/"logs"/f"HAAR_{group}.log")


In [ ]:
# E2 decision using fixed thresholds, if both groups completed.
from zipfile import ZipFile,ZIP_DEFLATED
if all((PROJECT/"HAAR"/g/"validation_result.json").is_file() for g in "BC"):
    keys=["box_map50_95_18","hard5_box_ap50_95","worst_box_ap50_95_18"]
    delta={}
    for g,t in [("B","C"),("C","B")]:
        h=json.loads((PROJECT/"HAAR"/g/"validation_result.json").read_text())
        b=json.loads((RGB_RUNS/g/"validation_result.json").read_text())
        delta[g+"_to_"+t]={k:h["evaluations"][t][k]-b["evaluations"][t][k] for k in keys}
    mean={k:sum(d[k] for d in delta.values())/2 for k in keys}
    passed=mean[keys[0]]>=.02 and mean[keys[1]]>=.02 and all(d[keys[0]]>0 and d[keys[2]]>=-.01 for d in delta.values())
    decision={"cross_deltas":delta,"cross_mean_delta":mean,"screen_pass":passed,"test_A_images_accessed":False}
    (PROJECT/"haar_comparison.json").write_text(json.dumps(decision,indent=2))
    print(json.dumps(decision,indent=2))
archive=WORK/"auliya-gc-diagnosis-report.zip"
with ZipFile(archive,"w",compression=ZIP_DEFLATED) as z:
    for p in sorted(PROJECT.rglob("*")):
        if p.is_file() and p.suffix.lower() not in {".pt",".part",".tmp"}:
            z.write(p,p.relative_to(PROJECT))
print("Upload:",archive)
print("Save Version for complete project:",PROJECT)
